# Replicating McDaniel et al. 2019 (sepsis in BioGears) on BioGears 8.2.0

_Investigation `mcdaniel-2019` — coder reproduction notebook._

**Question.** Does BioGears 8.2.0, driven through viva-biogears, reproduce McDaniel et al. (2019): sepsis progression by infection severity (Figs 4-7), the fluid challenge and antibiotic (Figs 9-10), and the REFRESH resuscitation protocols, on time and delayed (Figs 11-13)?

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-biogears/viva-biogears').is_dir():
    REPO = Path('/home/runner/work/viva-biogears/viva-biogears')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_biogears.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Sepsis progression by infection severity (McDaniel et al. 2019, Figs 4-7) (`mcdaniel-2019-sepsis`)

**Question.** Does BioGears 8.2.0, driven through viva-biogears, reproduce the sepsis progression McDaniel et al. (2019) report for BioGears' infection model: the inflammatory response ordered by infection severity (Figs 4-5), and the timing of hyperlactatemia, hypotension and septic shock in a severe untreated infection (Figs 6-7)?

**Objective.** Run biogears_sepsis for 48 h at 60 s updates (StandardMale@0s, Skin infection, MIC 16 mg/L, BioGears' InfectionModerate water and meal schedule) with no infection and with mild, moderate and severe infection, and compare the inflammatory and clinical readings.

**Hypothesis.** Peak pathogen and cytokine levels rise, and tissue integrity falls, from mild to moderate to severe infection. Only the severe arm decompensates: lactate exceeds 2 mM after about 20 h and SBP falls below 100 mmHg at about 26 h, as in the paper. Unlike the paper, MAP is not expected to reach 65 mmHg (septic shock) by 48 h on 8.2.0.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `control` | `viva_biogears.composites.sepsis.biogears_sepsis` | 2880 | severity=none, nutrition=infection_scenario, interval=60.0, n_steps=2880 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_biogears.composites.sepsis.biogears_sepsis`**

An infection in the real BioGears 8.2.0 engine (McDaniel et al. 2019): the acute inflammatory response, its clinical signs, the endothelial leak and lymph it drives, with BioGears' own meal and water schedule; emits every simulated minute by default.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_biogears_composites_sepsis_biogears_sepsis = {
    'severity'              : 'severe',   # Infection started at t = 0 ('none' for an uninfected control)
    'infection_location'    : 'Skin',   # Infection site (a label)
    'infection_mic_mg_per_L': 16.0,   # Pathogen minimum inhibitory concentration (the paper's 16 mg/L)
    'nutrition'             : 'infection_scenario',   # infection_scenario: BioGears' InfectionModerate schedule (water hourly, five meals in 48 h); none: no intake
    'iv_compound'           : 'Saline',   # IV fluid (the protocols' saline)
    'iv_rate_mL_per_min'    : 0.0,   # Infusion rate from t = 0 (no protocol only)
    'treatment'             : 'none',   # McDaniel et al. 2019 treatment protocol (viva_biogears.protocols): fluid_challenge (Fig 9), antibiotic (Fig 10), refresh_control and refresh_experimental (Figs 11-13); run to its end_h
    'challenge_h'           : 44.0,   # Saline challenge (2 x 500 mL) at
    'treatment_start_h'     : 46.0,   # Antibiotic and REFRESH start (58 for Fig 13's delayed arm)
    'treatment_duration_h'  : 6.0,   # REFRESH reassessment period
    'drug_readouts'         : None,   # Drugs whose concentrations are read (readings/treatment/<drug>); default Piperacillin and Norepinephrine when treated, none otherwise. Give an untreated arm the treated arms' list to record the same stores
    'state_file'            : 'StandardMale@0s.xml',   # BioGears state in <runtime>/states or the package's states/ (or an absolute path)
    'patient_file'          : '',   # Patient file in <runtime>/patients (or a path), stabilized once (about 40 s) and then cached; replaces state_file
    'patient_table'         : '',   # A Patients.csv (BioGears' template layout, one column per patient); with patient, the patient is generated from it, stabilized once and cached; replaces state_file
    'patient'               : '',   # The Name of a patient in patient_table
    'conditions'            : {},   # Chronic conditions applied while the patient stabilizes, e.g. {ChronicAnemia: {ReductionFactor: 0.3}}; with no patient_file, StandardMale.xml. A different patient, not a perturbation
    'substance_table'       : '',   # A Substances.csv, or a folder with Substances.csv and/or Compounds.csv (BioGears' template layout): their substances and compounds are added to the engine's substances
    'circuit_additions'     : {},   # New path name -> {source, target, resistance_mmHg_s_per_mL}: a resistance path between two FullCardiovascular nodes in different vascular compartments, added with its compartment link so substances follow. A perturbation: the body responds over time
    'interval'              : 60.0,   # Seconds per update
    'time_step_s'           : 0.02,   # BioGears' solver step (validated at 0.02 s; must be < 0.1 s)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mcdaniel-2019-sepsis ===
STUDY = 'mcdaniel-2019-sepsis'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_control = 2880
INTERVAL_control = 60.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_biogears_composites_sepsis_biogears_sepsis if 'ov_viva_biogears_composites_sepsis_biogears_sepsis' in dir() else None  # edits from the panel cell, if any
            spec_viva_biogears_composites_sepsis_biogears_sepsis = materialize_composite('viva_biogears.composites.sepsis.biogears_sepsis', _ov)
            _comp = Composite({'state': spec_viva_biogears_composites_sepsis_biogears_sepsis['state']}, core=core)
            _comp.run(STEPS_control)
        print('ran', 'control', f'(viva_biogears.composites.sepsis.biogears_sepsis) for', STEPS_control, 'steps')
    except Exception as _e:
        print(f"control not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**sepsis-progression**


In [ ]:
# sepsis-progression
show_viz(_render_one('local:SepsisProgression', {'inputs_map': {'time': 'global_time', 'blood_pathogen': 'readings.infection.inflammation.blood_pathogen', 'tissue_integrity': 'readings.infection.inflammation.tissue_integrity', 'mean_arterial_pressure': 'readings.infection.clinical.mean_arterial_pressure_mmHg', 'systolic_pressure': 'readings.infection.clinical.systolic_pressure_mmHg', 'lactate': 'readings.infection.clinical.lactate_mmol_per_L', 'blood_volume': 'readings.infection.clinical.blood_volume_mL'}}, RUNS_DB, STUDY_YAML))

### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Fig 4-5: the inflammatory response (pathogen, TNF, IL-6, IL-10, NO) rises with infection severity | readings.infection.inflammation.*, peak per arm |  | Available (readouts recorded) |
| Figs 6-7: severe infection, lactate > 2 mM after about 20 h | readings.infection.clinical.lactate_mmol_per_L |  | Available (readouts recorded) |
| Figs 6-7: SBP < 100 mmHg at about 26 h, sepsis at about 30 h, septic shock (MAP < 65, lactate > 2 mM) at about 45 h | readings.infection.clinical (SBP, MAP, lactate, respiration rate) |  | Available (readouts recorded) |
| Fig 6: SVR falls toward 60 % of baseline | readings.infection.clinical.systemic_vascular_resistance_mmHg_s_per_mL |  | Available (readouts recorded) |


## Study: Fluid challenge and antibiotic in severe sepsis (McDaniel et al. 2019, Figs 9-10) (`mcdaniel-2019-treatment`)

**Question.** In the severe infection, what do the paper's 1000 mL saline challenge at 44 h (Fig 9) and 4.5 g piperacillin/tazobactam at 46 h (Fig 10) do to volume, pressure and the pathogen?

**Objective.** Starting from the severe infection's state saved at 44 h, run biogears_sepsis (60 s updates) to 52 h untreated, with treatment fluid_challenge (2 x 500 mL saline at 50 mL/min at 44 and 44.5 h) and with treatment antibiotic (the challenge, then 20 mL PiperacillinTazobactam over 30 min at 46 h).

**Hypothesis.** The challenge raises blood volume and pressure above the untreated arm within the hour it is given; the antibiotic's plasma level exceeds the pathogen's MIC (16 mg/L) and the blood pathogen falls below the challenge-only arm's by 52 h.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `untreated` | `viva_biogears.composites.sepsis.biogears_sepsis` | 480 | severity=severe, nutrition=infection_scenario, interval=60.0, treatment=none, drug_readouts=['Piperacillin', 'Norepinephrine'], state_file=StandardMale@severe-infection-44h.xml, n_steps=480 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_biogears.composites.sepsis.biogears_sepsis`**

An infection in the real BioGears 8.2.0 engine (McDaniel et al. 2019): the acute inflammatory response, its clinical signs, the endothelial leak and lymph it drives, with BioGears' own meal and water schedule; emits every simulated minute by default.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_biogears_composites_sepsis_biogears_sepsis = {
    'severity'              : 'severe',   # Infection started at t = 0 ('none' for an uninfected control)
    'infection_location'    : 'Skin',   # Infection site (a label)
    'infection_mic_mg_per_L': 16.0,   # Pathogen minimum inhibitory concentration (the paper's 16 mg/L)
    'nutrition'             : 'infection_scenario',   # infection_scenario: BioGears' InfectionModerate schedule (water hourly, five meals in 48 h); none: no intake
    'iv_compound'           : 'Saline',   # IV fluid (the protocols' saline)
    'iv_rate_mL_per_min'    : 0.0,   # Infusion rate from t = 0 (no protocol only)
    'treatment'             : 'none',   # McDaniel et al. 2019 treatment protocol (viva_biogears.protocols): fluid_challenge (Fig 9), antibiotic (Fig 10), refresh_control and refresh_experimental (Figs 11-13); run to its end_h
    'challenge_h'           : 44.0,   # Saline challenge (2 x 500 mL) at
    'treatment_start_h'     : 46.0,   # Antibiotic and REFRESH start (58 for Fig 13's delayed arm)
    'treatment_duration_h'  : 6.0,   # REFRESH reassessment period
    'drug_readouts'         : None,   # Drugs whose concentrations are read (readings/treatment/<drug>); default Piperacillin and Norepinephrine when treated, none otherwise. Give an untreated arm the treated arms' list to record the same stores
    'state_file'            : 'StandardMale@0s.xml',   # BioGears state in <runtime>/states or the package's states/ (or an absolute path)
    'patient_file'          : '',   # Patient file in <runtime>/patients (or a path), stabilized once (about 40 s) and then cached; replaces state_file
    'patient_table'         : '',   # A Patients.csv (BioGears' template layout, one column per patient); with patient, the patient is generated from it, stabilized once and cached; replaces state_file
    'patient'               : '',   # The Name of a patient in patient_table
    'conditions'            : {},   # Chronic conditions applied while the patient stabilizes, e.g. {ChronicAnemia: {ReductionFactor: 0.3}}; with no patient_file, StandardMale.xml. A different patient, not a perturbation
    'substance_table'       : '',   # A Substances.csv, or a folder with Substances.csv and/or Compounds.csv (BioGears' template layout): their substances and compounds are added to the engine's substances
    'circuit_additions'     : {},   # New path name -> {source, target, resistance_mmHg_s_per_mL}: a resistance path between two FullCardiovascular nodes in different vascular compartments, added with its compartment link so substances follow. A perturbation: the body responds over time
    'interval'              : 60.0,   # Seconds per update
    'time_step_s'           : 0.02,   # BioGears' solver step (validated at 0.02 s; must be < 0.1 s)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mcdaniel-2019-treatment ===
STUDY = 'mcdaniel-2019-treatment'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_untreated = 480
INTERVAL_untreated = 60.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_biogears_composites_sepsis_biogears_sepsis if 'ov_viva_biogears_composites_sepsis_biogears_sepsis' in dir() else None  # edits from the panel cell, if any
            spec_viva_biogears_composites_sepsis_biogears_sepsis = materialize_composite('viva_biogears.composites.sepsis.biogears_sepsis', _ov)
            _comp = Composite({'state': spec_viva_biogears_composites_sepsis_biogears_sepsis['state']}, core=core)
            _comp.run(STEPS_untreated)
        print('ran', 'untreated', f'(viva_biogears.composites.sepsis.biogears_sepsis) for', STEPS_untreated, 'steps')
    except Exception as _e:
        print(f"untreated not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**treatment-response**


In [ ]:
# treatment-response
show_viz(_render_one('local:SepsisTreatmentResponse', {'inputs_map': {'time': 'global_time', 'protocol_code': 'protocol.code', 'mean_arterial_pressure': 'readings.infection.clinical.mean_arterial_pressure_mmHg', 'systolic_pressure': 'readings.infection.clinical.systolic_pressure_mmHg', 'mean_urine_output': 'readings.infection.clinical.mean_urine_output_mL_per_min', 'blood_pathogen': 'readings.infection.inflammation.blood_pathogen', 'saline_given': 'protocol.saline_given_mL', 'branch_h': 'protocol.branch_h', 'treatment_start_h': 'protocol.treatment_start_h'}}, RUNS_DB, STUDY_YAML))

### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Fig 9: the 1000 mL saline challenge transiently raises SBP and urine output | readings.infection.clinical.systolic_pressure_mmHg, mean_urine_output_mL_per_min, 44-46 h |  | Available (readouts recorded) |
| Fig 10: piperacillin rises above the MIC (16 mg/L) during the 30 min infusion and the pathogen declines | readings.treatment.Piperacillin.plasma_concentration_ug_per_mL; readings.infection.inflammation.blood_pathogen |  | Available (readouts recorded) |


## Study: REFRESH resuscitation protocols in severe sepsis (McDaniel et al. 2019, Figs 11-13) (`mcdaniel-2019-refresh`)

**Question.** Do the paper's two REFRESH-trial protocols, liberal fluid boluses (control) and restricted fluid with norepinephrine (experimental), reach the resuscitation goals in the severe infection, with how much fluid, and what does a 12 h delay do to the control protocol (Fig 13)?

**Objective.** Starting from the severe infection's state saved at 44 h, run biogears_sepsis (60 s updates) with treatment refresh_control and refresh_experimental (challenge at 44 h, antibiotic and protocol from 46 to 52 h), and refresh_control delayed by 12 h (challenge at 56 h, protocol from 58 to 64 h).

**Hypothesis.** Both protocols raise SBP above 90 and MAP above 65 mmHg by the end of their 6 h; the experimental protocol does so with less saline than the control; the same control protocol started 12 h later meets a more hypotensive patient and gives more saline.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `refresh-control` | `viva_biogears.composites.sepsis.biogears_sepsis` | 480 | severity=severe, nutrition=infection_scenario, interval=60.0, treatment=refresh_control, state_file=StandardMale@severe-infection-44h.xml, n_steps=480 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_biogears.composites.sepsis.biogears_sepsis`**

An infection in the real BioGears 8.2.0 engine (McDaniel et al. 2019): the acute inflammatory response, its clinical signs, the endothelial leak and lymph it drives, with BioGears' own meal and water schedule; emits every simulated minute by default.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_biogears_composites_sepsis_biogears_sepsis = {
    'severity'              : 'severe',   # Infection started at t = 0 ('none' for an uninfected control)
    'infection_location'    : 'Skin',   # Infection site (a label)
    'infection_mic_mg_per_L': 16.0,   # Pathogen minimum inhibitory concentration (the paper's 16 mg/L)
    'nutrition'             : 'infection_scenario',   # infection_scenario: BioGears' InfectionModerate schedule (water hourly, five meals in 48 h); none: no intake
    'iv_compound'           : 'Saline',   # IV fluid (the protocols' saline)
    'iv_rate_mL_per_min'    : 0.0,   # Infusion rate from t = 0 (no protocol only)
    'treatment'             : 'none',   # McDaniel et al. 2019 treatment protocol (viva_biogears.protocols): fluid_challenge (Fig 9), antibiotic (Fig 10), refresh_control and refresh_experimental (Figs 11-13); run to its end_h
    'challenge_h'           : 44.0,   # Saline challenge (2 x 500 mL) at
    'treatment_start_h'     : 46.0,   # Antibiotic and REFRESH start (58 for Fig 13's delayed arm)
    'treatment_duration_h'  : 6.0,   # REFRESH reassessment period
    'drug_readouts'         : None,   # Drugs whose concentrations are read (readings/treatment/<drug>); default Piperacillin and Norepinephrine when treated, none otherwise. Give an untreated arm the treated arms' list to record the same stores
    'state_file'            : 'StandardMale@0s.xml',   # BioGears state in <runtime>/states or the package's states/ (or an absolute path)
    'patient_file'          : '',   # Patient file in <runtime>/patients (or a path), stabilized once (about 40 s) and then cached; replaces state_file
    'patient_table'         : '',   # A Patients.csv (BioGears' template layout, one column per patient); with patient, the patient is generated from it, stabilized once and cached; replaces state_file
    'patient'               : '',   # The Name of a patient in patient_table
    'conditions'            : {},   # Chronic conditions applied while the patient stabilizes, e.g. {ChronicAnemia: {ReductionFactor: 0.3}}; with no patient_file, StandardMale.xml. A different patient, not a perturbation
    'substance_table'       : '',   # A Substances.csv, or a folder with Substances.csv and/or Compounds.csv (BioGears' template layout): their substances and compounds are added to the engine's substances
    'circuit_additions'     : {},   # New path name -> {source, target, resistance_mmHg_s_per_mL}: a resistance path between two FullCardiovascular nodes in different vascular compartments, added with its compartment link so substances follow. A perturbation: the body responds over time
    'interval'              : 60.0,   # Seconds per update
    'time_step_s'           : 0.02,   # BioGears' solver step (validated at 0.02 s; must be < 0.1 s)
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mcdaniel-2019-refresh ===
STUDY = 'mcdaniel-2019-refresh'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_refresh_control = 480
INTERVAL_refresh_control = 60.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_biogears_composites_sepsis_biogears_sepsis if 'ov_viva_biogears_composites_sepsis_biogears_sepsis' in dir() else None  # edits from the panel cell, if any
            spec_viva_biogears_composites_sepsis_biogears_sepsis = materialize_composite('viva_biogears.composites.sepsis.biogears_sepsis', _ov)
            _comp = Composite({'state': spec_viva_biogears_composites_sepsis_biogears_sepsis['state']}, core=core)
            _comp.run(STEPS_refresh_control)
        print('ran', 'refresh-control', f'(viva_biogears.composites.sepsis.biogears_sepsis) for', STEPS_refresh_control, 'steps')
    except Exception as _e:
        print(f"refresh-control not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**treatment-response**


In [ ]:
# treatment-response
show_viz(_render_one('local:SepsisTreatmentResponse', {'inputs_map': {'time': 'global_time', 'protocol_code': 'protocol.code', 'mean_arterial_pressure': 'readings.infection.clinical.mean_arterial_pressure_mmHg', 'systolic_pressure': 'readings.infection.clinical.systolic_pressure_mmHg', 'mean_urine_output': 'readings.infection.clinical.mean_urine_output_mL_per_min', 'blood_pathogen': 'readings.infection.inflammation.blood_pathogen', 'saline_given': 'protocol.saline_given_mL', 'branch_h': 'protocol.branch_h', 'treatment_start_h': 'protocol.treatment_start_h'}}, RUNS_DB, STUDY_YAML))

### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Figs 11-12: both REFRESH protocols reach SBP > 90 and MAP > 65 mmHg; the experimental (norepinephrine) protocol with less fluid | readings.infection.clinical (SBP, MAP); protocol.saline_given_mL |  | Available (readouts recorded) |
| Fig 13: the control protocol delayed 12 h needs more fluid | protocol.saline_given_mL, refresh-control-delayed vs refresh-control |  | Available (readouts recorded) |
